# 2 · Water classification and shoreline extraction


For every scene we classify water against land and recover the shoreline as a polyline. The method follows three classical ideas:

1. **Modified Normalised Difference Water Index** — MNDWI `(green − SWIR) / (green + SWIR)` uses the short-wave infrared band instead of NIR, which suppresses built-up land and vegetation responses that contaminate the original NDWI of McFeeters (1996) [6] (Xu 2006 [5]). Open water has strongly positive MNDWI, sand beaches are near zero or negative.
2. **Otsu thresholding** — the index histogram of clear pixels is bimodal (water / non-water), and Otsu's criterion (1979) [7] picks the threshold that minimises intra-class variance. We compute it per scene so the method adapts to sun glint, haze and seasonal water colour.
3. **Cloud masking** — the Sentinel-2 SCL layer flags nodata, cloud shadow, medium/high cloud and cirrus; only valid surface pixels enter the classification.

**From mask to line.** The Atlantic Ocean is the largest connected water component. In every crop window the open ocean reaches the southern image edge, so for each image column that truly crosses the beach the shoreline is simply the bottom edge of the clear-land class. Columns whose bottom edge belongs to a thin land finger, a lagoon or an inlet mouth are rejected by a two-stage robust median filter applied along the track; the surviving points are Douglas–Peucker simplified and reprojected to WGS84. The result is one clean polyline per scene, saved to `data/shorelines_all.geojson`.


```bash
cd .. && PYTHONPATH=. python3 scripts/03_extract_shorelines.py
```


In [ ]:
# The full implementation lives in scripts/03_extract_shorelines.py; the
# core ingredients are shown here for transparency.

import numpy as np
import rasterio
from scipy import ndimage
from skimage.filters import threshold_otsu

# 1. read 20 m green and SWIR-1, cropped to the study bbox
with rasterio.open("data/raw/31NEH/2025-01-24/B03.jp2") as src:
    green = src.read(1).astype(np.float32) * 1e-4
    transform = src.transform
with rasterio.open("data/raw/31NEH/2025-01-24/B11.jp2") as src:
    swir = src.read(1).astype(np.float32) * 1e-4
with rasterio.open("data/raw/31NEH/2025-01-24/SCL.jp2") as src:
    scl = src.read(1)

# 2. clear-pixel mask from the SCL layer
excluded = {0, 2, 3, 8, 9, 10, 11}
clear = ~np.isin(scl, list(excluded))

# 3. MNDWI + Otsu threshold
mndwi = (green - swir) / np.where(green + swir > 0, green + swir, 1e-6)
thr = threshold_otsu(mndwi[clear])
water = clear & (mndwi > max(thr, -0.15))

# 4. morphological cleanup (remove salt, close gaps) with edge restore
water = ndimage.binary_opening(water, structure=np.ones((3, 3)))
water = ndimage.binary_closing(water, structure=np.ones((7, 7)))

# 5. ocean = largest connected water component; coast = column-wise
#    bottom edge of the land class where the ocean fills the window bottom
lab, n = ndimage.label(water)
sizes = ndimage.sum(water, lab, range(1, n + 1))
ocean = lab == (int(np.argmax(sizes)) + 1)
print(f"water pixels: {water.sum() / 1e6:.1f} M; "
      f"ocean area: {ocean.sum() / 1e6:.1f} M px")

